In [1]:
import jax
import jax.numpy as jnp
import numpy as onp
import jax_sn
%load_ext autoreload
%autoreload 2
jax.config.update("jax_enable_x64", True)

In [2]:
from jax_sn.tools.setup_test_mgop import setup_slab_mgop, make_xs_test, setup_layered_slab_mgop

In [3]:

def _build_base_mgop():
    """Build the reference mgop with default (Vmap) operators. Expensive — call once."""
    xs = [make_xs_test(m, n_groups=2, l_order=1) for m in ["fuel", "moderator", "reflector"]]
    return setup_layered_slab_mgop(
        dim=3,
        degree=1,
        tn_quadrature_set=3,
        region_sizes=[1.0, 4.0, 2.0],
        region_resolutions=[2, 1, 3],
        region_xs=xs,
        region_sources=[1.0, 0.5, 0.2],
        transverse_specs=[(2.0, 2), (1.523, 3)],
    )



In [4]:
from jax_sn.operators.transport_operator.linear_tetrahedron_warp_kernel import WarpSweep
onp.random.seed(2042026)

base_mgop = _build_base_mgop()

b_apply = jnp.array(onp.random.random_sample(base_mgop.transport_operator.out_structure().shape))
xs = base_mgop.domain.cross_sections_indexed.map_to_full_array(0)

ws =  WarpSweep.from_transport_operator(base_mgop.transport_operator)

Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "aarch64"
     "cuda:0"   : "NVIDIA GB10" (120 GiB, sm_121, mempool enabled)
   Kernel cache:
     /home/spark-bcdd/.cache/warp/1.12.0


In [7]:
result = ws.compute_single_angle(xs.total, 0, b_apply[0])

In [10]:
onp.testing.assert_allclose(result, base_mgop.transport_operator[0].inv(b_apply[0]))

In [23]:
import pyvista as pv
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
plotter  = pv.Plotter()
plotter.add_mesh(mesh_to_pyvista_mesh(mgop.domain.vertices, mgop.domain.connectivity), scalars = mgop.domain.cross_sections_indexed.indices)
plotter.show()

Widget(value='<iframe src="http://localhost:44001/index.html?ui=P_0xe874841174a0_5&reconnect=auto" class="pyvi…